# 04 Logistic Regression With Basic Image Features

This is the first real supervised image-feature model. Logistic Regression is a simple linear classifier, so this notebook tests whether basic intensity, entropy, and edge-density features from cropped SEM image content can improve on the majority-class dummy baseline.

## 1. Project setup

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

## 2. Imports

In [ ]:
import os
os.environ.setdefault("MPLCONFIGDIR", "/tmp/mse446_matplotlib")

import matplotlib.pyplot as plt
import pandas as pd
from sklearn.metrics import ConfusionMatrixDisplay

from src.extract_basic_features import BASIC_FEATURE_COLUMNS, FEATURES_CSV
from src.train_logistic_regression import (
    CLASS_LABELS,
    CONFUSION_MATRIX_FIGURE,
    SCORES_CSV,
    choose_group_split,
    evaluate_predictions,
    load_features,
    make_model,
    print_dummy_comparison,
    save_confusion_matrix_figure,
)

## 3. Paths and configuration

In [ ]:
print(f"Basic features CSV: {FEATURES_CSV}")
print(f"Scores CSV: {SCORES_CSV}")
print(f"Confusion matrix figure: {CONFUSION_MATRIX_FIGURE}")

## 4. Load metadata

In [ ]:
features = load_features(FEATURES_CSV)
features.head()

## 5. Sanity checks

In [ ]:
print(f"Rows: {len(features)}")
print("Label counts:")
print(features["label"].value_counts().sort_index())
print(f"Image feature columns: {len(BASIC_FEATURE_COLUMNS)}")
print(BASIC_FEATURE_COLUMNS)
assert set(BASIC_FEATURE_COLUMNS).issubset(features.columns)
assert features["area_group"].notna().all()

## 6. Main analysis

Only image-derived feature columns are used as predictors. Metadata columns such as label, sample, area, voltage, working distance, and magnification are used only for labels, splitting, and audit checks.

In [ ]:
train_idx, test_idx, selected_seed, split_distance = choose_group_split(features)

groups = features["area_group"]
assert set(groups.iloc[train_idx]).isdisjoint(set(groups.iloc[test_idx]))

X = features[BASIC_FEATURE_COLUMNS]
y = features["label"]
X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]
y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

model = make_model()
model.fit(X_train, y_train)
y_train_pred = model.predict(X_train)
y_test_pred = model.predict(X_test)

scores, report, matrix = evaluate_predictions(
    y_train,
    y_test,
    y_train_pred,
    y_test_pred,
    selected_seed,
    split_distance,
)
scores

## 7. Results/figures

Balanced accuracy and macro F1 matter more than raw accuracy because the dataset has many more ordered images than disordered images. A useful model should improve disordered recall compared with the dummy baseline.

In [ ]:
scores.to_csv(SCORES_CSV, index=False)
figure_path = save_confusion_matrix_figure(y_test, y_test_pred, CONFUSION_MATRIX_FIGURE)

print("Selected split seed:", selected_seed)
print("Train label counts:")
print(y_train.value_counts().sort_index())
print("Test label counts:")
print(y_test.value_counts().sort_index())
print("\nPer-class precision/recall/F1:")
print(report)
print_dummy_comparison(scores)
print(f"Saved scores to {SCORES_CSV}")
print(f"Saved confusion matrix to {figure_path}")

ConfusionMatrixDisplay.from_predictions(
    y_test,
    y_test_pred,
    labels=CLASS_LABELS,
    cmap="Blues",
    colorbar=False,
)
plt.title("Logistic regression basic")
plt.show()

## 8. Notes for report

- This is the first model that uses actual image content as predictors.
- The feature set is intentionally simple and excludes HOG, graph features, and CNN representations.
- Metadata is not used as model input; it is retained only for splitting and interpretation.